# 05. Flower Classification using Pretrained ResNet152 model

The project utilizes a dataset containing images of various flower species. The dataset is divided into a training set and a test set, each labeled with the corresponding flower species. There are 102 flower species.

In [21]:
# Import libraries
from pathlib import Path
import matplotlib.pyplot as plt
import torch
import torchvision.models
from torch import nn
from torchinfo import summary
from torchvision import datasets
from torchvision import transforms
from torchvision.models import ResNet152_Weights

from src.data_loader import create_dataloaders
from src.models.vgg_16 import VGG16
from src.utils import set_device_agnostic_mode
from src.training import train_model

In [2]:
# Initialize training parameters and get device
train_dir = 'data/train'
val_dir = 'data/val'

# Get a device to use for training/inference
device = set_device_agnostic_mode()

In [7]:
# Create ResNet152 CNN model

# Get resnet152 params, weights, and transforms
COLOR_CHANNELS = 3
BATCH_SIZE = 32
EPOCHS = 50

resnet152_weights = ResNet152_Weights.IMAGENET1K_V2
auto_transforms = resnet152_weights.transforms()

# Create training and validation data loaders
train_dataloader, val_dataloader, classes = create_dataloaders(train_dir, val_dir, 
                                                               auto_transforms, auto_transforms, BATCH_SIZE)

# Initialize Resnet 152 model and fine tune classification layer
resnet152_model = torchvision.models.resnet152(weights=resnet152_weights)

for name, layer in resnet152_model.named_children():
    if name not in ['fc']:
        for param in layer.parameters():
            param.requires_grad = False

resnet152_model.fc = nn.Linear(in_features=2048, out_features=len(classes), bias=True)
print(summary(resnet152_model,
        input_size=(BATCH_SIZE, COLOR_CHANNELS, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"]))
resnet152_model = resnet152_model.to(device)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
ResNet                                   [32, 3, 224, 224]         [32, 102]                 --                        Partial
├─Conv2d: 1-1                            [32, 3, 224, 224]         [32, 64, 112, 112]        (9,408)                   False
├─BatchNorm2d: 1-2                       [32, 64, 112, 112]        [32, 64, 112, 112]        (128)                     False
├─ReLU: 1-3                              [32, 64, 112, 112]        [32, 64, 112, 112]        --                        --
├─MaxPool2d: 1-4                         [32, 64, 112, 112]        [32, 64, 56, 56]          --                        --
├─Sequential: 1-5                        [32, 64, 56, 56]          [32, 256, 56, 56]         --                        False
│    └─Bottleneck: 2-1                   [32, 64, 56, 56]          [32, 256, 56, 56]         --                        False


In [8]:
# Train ResNet152 CNN model

# Create optimizer and loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet152_model.parameters(), lr=0.0001)

# Train model
train_model(
    EPOCHS,
    len(classes),
    resnet152_model,
    train_dataloader,
    val_dataloader,
    loss_fn,
    optimizer,
    device
)

Epoch 1, Train Loss 4.156656742095947, Train Accuracy 0.10093912482261658, Val Loss 4.2178754806518555, Val Accuracy 0.0849609375
Epoch 2, Train Loss 3.5155158042907715, Train Accuracy 0.3206930160522461, Val Loss 3.7940120697021484, Val Accuracy 0.2060546875
Epoch 3, Train Loss 3.0310332775115967, Train Accuracy 0.4863665997982025, Val Loss 3.4619152545928955, Val Accuracy 0.3408203125
Epoch 4, Train Loss 2.6175029277801514, Train Accuracy 0.6005181670188904, Val Loss 3.101947069168091, Val Accuracy 0.4455915093421936
Epoch 5, Train Loss 2.2811007499694824, Train Accuracy 0.6955634355545044, Val Loss 2.843445301055908, Val Accuracy 0.5005580186843872
Epoch 6, Train Loss 1.9979451894760132, Train Accuracy 0.7563471794128418, Val Loss 2.634256601333618, Val Accuracy 0.5883091688156128
Epoch 7, Train Loss 1.7626341581344604, Train Accuracy 0.807253897190094, Val Loss 2.385254383087158, Val Accuracy 0.6450892686843872
Epoch 8, Train Loss 1.573642611503601, Train Accuracy 0.844332933425903

In [9]:
# Save ResNet152 model

# Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Create model save path
MODEL_NAME = "05_flower_classification_resnet152.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=resnet152_model.state_dict(), f=MODEL_SAVE_PATH)

Saving model to: models/05_flower_classification_resnet152.pth
